In [1]:
import subprocess
import re
import pandas as pd
import shutil
from tqdm import tqdm
import openpyxl
import os
import gc

In [2]:
tree = "propose_tree"

In [3]:
# quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"

# # Compile
# subprocess.run(
#     [f"{quartus_bin}\\quartus_sh.exe", "--flow", "compile", "generic_propose_tree"],
#         cwd=rf"C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\v256",
#     check=True
# )

# # Timing
# subprocess.run(
#     [f"{quartus_bin}\\quartus_sta.exe", "-t", "script.tcl"],
#     cwd=rf"C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\v256",
#     check=True
# )

## Diretorios

In [8]:
quartus_bin = r"C:\altera_standard\25.1std\quartus\bin64"
base_dir = rf"C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal"

m_values = [16, 32, 64, 128, 256, 512]


# m_values = [256]
# n_values = [256] 

## Functions

In [4]:
# =========================
# altera n no VHDL
# =========================
def set_n(vhdl_file, n):
    with open(vhdl_file, "r", encoding="utf-8", errors="ignore") as f:
        text = f.read()

    text = re.sub(
        r"n\s*:\s*integer\s*:=\s*\d+",
        f"n : integer := {n}",
        text
    )

    with open(vhdl_file, "w", encoding="utf-8") as f:
        f.write(text)


# =========================
# acha nome do projeto .qpf
# =========================
def find_project_name(project_dir):
    qpf_files = [f for f in os.listdir(project_dir) if f.endswith(".qpf")]

    if not qpf_files:
        raise FileNotFoundError(f"Nenhum arquivo .qpf encontrado em {project_dir}")

    return os.path.splitext(qpf_files[0])[0]


# =========================
# roda quartus
# =========================
def run_quartus(project_dir):
    shutil.rmtree(os.path.join(project_dir, "db"), ignore_errors=True)
    shutil.rmtree(os.path.join(project_dir, "incremental_db"), ignore_errors=True)

    project_name = find_project_name(project_dir)

    try:
        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sh.exe"),
                "--flow",
                "compile",
                project_name
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )

        subprocess.run(
            [
                os.path.join(quartus_bin, "quartus_sta.exe"),
                "-t",
                "script.tcl"
            ],
            cwd=project_dir,
            check=True,
            capture_output=True,
            text=True
        )
    except subprocess.CalledProcessError as e:
        raise RuntimeError(
            f"Quartus failed in {project_dir} with return code {e.returncode}\n"
            f"stdout:\n{e.stdout}\n"
            f"stderr:\n{e.stderr}"
        ) from e


# =========================
# parse dos paths
# =========================
def parse_paths(file):
    paths = []
    current = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Path #\d+: Delay is\s+([\d\.]+)", line)
            if m:
                if current:
                    paths.append(current)

                current = {
                    "delay": float(m.group(1)),
                    "ic": None,
                    "cell": None
                }
                continue

            if current is None:
                continue

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["ic"] = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                current["cell"] = float(m.group(1))

    if current:
        paths.append(current)

    df = pd.DataFrame(paths)

    if df.empty:
        return df

    return df.dropna()


# =========================
# calcula métricas
# =========================
def compute_metrics(df):
    critical = df.loc[df["delay"].idxmax()]

    return {
        "Delay_CP": critical["delay"],
        "IC_CP": critical["ic"],
        "CELL_CP": critical["cell"],

        "Delay_MAX": df["delay"].max(),
        "IC_MAX": df["ic"].max(),
        "CELL_MAX": df["cell"].max(),

        "Delay_MEAN": df["delay"].mean(),
        "IC_MEAN": df["ic"].mean(),
        "CELL_MEAN": df["cell"].mean()
    }


# =========================
# extrai caminho crítico
# =========================
def extract_critical_path(file):
    delay = None
    ic = None
    cell = None
    from_node = None
    to_node = None

    with open(file, "r", encoding="utf-8", errors="ignore") as f:
        for line in f:

            m = re.search(r"Delay is\s+([\d\.]+)", line)
            if m:
                delay = float(m.group(1))

            m = re.search(r";\s*From Node\s*;\s*(.*?)\s*;", line)
            if m:
                from_node = m.group(1)

            m = re.search(r";\s*To Node\s*;\s*(.*?)\s*;", line)
            if m:
                to_node = m.group(1)

            m = re.search(r";\s*IC\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                ic = float(m.group(1))

            m = re.search(r";\s*Cell\s*;.*?;\s*\d+\s*;\s*([\d\.]+)", line)
            if m:
                cell = float(m.group(1))

    return {
        "delay": delay,
        "ic": ic,
        "cell": cell,
        "from": from_node,
        "to": to_node
    }


In [ ]:
# =========================
# loop principal
# =========================
resultados = []

for num_vec in tqdm(m_values):

    project_dir = os.path.join(base_dir, f"v{num_vec}")
    vhdl_file = os.path.join(project_dir, "vhdl", f"sum{num_vec}xn.vhd")
    report_file = os.path.join(project_dir, "io_paths.txt")
    worst_file = os.path.join(project_dir, "worst_path.txt")

    resultados_m = []

    print(f"\n==============================")
    print(f" Rodando projeto com {num_vec} vetores")
    print(f" Casos: {num_vec}x{num_vec} e {num_vec}x{2*num_vec}")
    print(f"==============================")

    # roda N e 2N
    for n in [num_vec, 2*num_vec]:

        df = None
        metrics = None
        cp = None

        try:
            print(f"\nRodando caso {num_vec}x{n}")

            set_n(vhdl_file, n)
            run_quartus(project_dir)

            df = parse_paths(report_file)

            if df.empty:
                raise ValueError("Nenhum path encontrado em io_paths.txt")

            metrics = compute_metrics(df)
            cp = extract_critical_path(worst_file)

            linha = {
                "num_vec": num_vec,
                "n": n,
                "caso": f"{num_vec}x{n}",

                "Delay_CP": cp["delay"],
                "IC_CP": cp["ic"],
                "CELL_CP": cp["cell"],

                "From_Node": cp["from"],
                "To_Node": cp["to"],

                "Delay_MAX": metrics["Delay_MAX"],
                "IC_MAX": metrics["IC_MAX"],
                "CELL_MAX": metrics["CELL_MAX"],

                "Delay_MEAN": metrics["Delay_MEAN"],
                "IC_MEAN": metrics["IC_MEAN"],
                "CELL_MEAN": metrics["CELL_MEAN"],
            }

            resultados.append(linha)
            resultados_m.append(linha)

        except Exception as e:
            print(f"Erro em num_vec={num_vec}, n={n}: {e}")
            continue

        finally:
            for name in ("df", "metrics", "cp"):
                globals().pop(name, None)
            gc.collect()

    # salva um arquivo separado para cada número de vetores
    if resultados_m:
        df_m = pd.DataFrame(resultados_m).sort_values("n")

        csv_m = os.path.join(base_dir, f"timing_v{num_vec}_N_2N.csv")
        xlsx_m = os.path.join(base_dir, f"timing_v{num_vec}_N_2N.xlsx")

        df_m.to_csv(csv_m, index=False)
        df_m.to_excel(xlsx_m, index=False)

        print(f"\nArquivos salvos para {num_vec} vetores:")
        print(csv_m)
        print(xlsx_m)


# =========================
# salva resultado geral
# =========================
df_all = pd.DataFrame(resultados).sort_values(["num_vec", "n"])

csv_all = os.path.join(base_dir, "timing_N_2N.csv")
xlsx_all = os.path.join(base_dir, "timing_N_2N.xlsx")

df_all.to_csv(csv_all, index=False)
df_all.to_excel(xlsx_all, index=False)

print("\nArquivos gerais salvos:")
print(csv_all)
print(xlsx_all)

  0%|          | 0/6 [00:00<?, ?it/s]


 Rodando projeto com 16 vetores
 Casos: 16x16 e 16x32

Rodando caso 16x16

Rodando caso 16x32


 17%|█▋        | 1/6 [04:05<20:27, 245.42s/it]


Arquivos salvos para 16 vetores:
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\timing_v16_N_2N.csv
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\timing_v16_N_2N.xlsx

 Rodando projeto com 32 vetores
 Casos: 32x32 e 32x64

Rodando caso 32x32

Rodando caso 32x64


 33%|███▎      | 2/6 [08:20<16:45, 251.39s/it]

...


Arquivos salvos para 128 vetores:
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\timing_v128_N_2N.csv
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\timing_v128_N_2N.xlsx

 Rodando projeto com 256 vetores
 Casos: 256x256 e 256x512

Rodando caso 256x256

Rodando caso 256x512


 83%|████████▎ | 5/6 [2:27:03<48:22, 2902.66s/it]


Arquivos salvos para 256 vetores:
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\timing_v256_N_2N.csv
C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\timing_v256_N_2N.xlsx

 Rodando projeto com 512 vetores
 Casos: 512x512 e 512x1024

Rodando caso 512x512

Rodando caso 512x1024


100%|██████████| 6/6 [16:31:05<00:00, 9910.86s/it] 

Erro em num_vec=512, n=1024: Quartus failed in C:\Users\Rafa\Desktop\HCR\Generics_Tree\propose_tree_diagonal\v512 with return code 3
stdout:
Info: *******************************************************************
Info: Running Quartus Prime Shell
    Info: Version 25.1std.0 Build 1129 10/21/2025 SC Standard Edition
    Info: Copyright (C) 2025  Altera Corporation. All rights reserved.
    Info: Your use of Altera Corporation's design tools, logic functions 
    Info: and other software and tools, and any partner logic 
    Info: functions, and any output files from any of the foregoing 
    Info: (including device programming or simulation files), and any 
    Info: associated documentation or information are expressly subject 
    Info: to the terms and conditions of the Altera Program License 
    Info: Subscription Agreement, the Altera Quartus Prime License Agreement,
    Info: the Altera IP License Agreement, or other applicable license
    Info: agreement, including, without li

In [12]:
# =========================

# salva resultado geral
# =========================
final_df = pd.DataFrame(resultados).sort_values(["num_vec", "n"])

csv_final = os.path.join(base_dir, "timing_all_vectors.csv")
xlsx_final = os.path.join(base_dir, "timing_all_vectors.xlsx")

final_df.to_csv(csv_final, index=False)
final_df.to_excel(xlsx_final, index=False)

print("\nResultado final:")
print(final_df)

print("\nArquivos gerais salvos:")
print(csv_final)
print(xlsx_final)


Resultado final:
    num_vec    n     caso  Delay_CP   IC_CP  CELL_CP From_Node   To_Node  \
0        16   16    16x16     5.803   2.912    2.891     a7[2]   sum[15]   
1        16   32    16x32     8.572   5.907    2.665     a6[2]   sum[16]   
2        32   32    32x32     8.963   4.695    4.268    a17[4]   sum[31]   
3        32   64    32x64    10.999   6.655    4.344     a1[3]   sum[38]   
4        64   64    64x64    12.426   7.831    4.595    a27[0]   sum[51]   
5        64  128   64x128    24.160  20.876    3.284    a6[52]   sum[99]   
6       128  128  128x128    20.414  14.300    6.114    a64[7]  sum[125]   
7       128  256  128x256    28.416  21.607    6.809     a1[3]  sum[201]   
8       256  256  256x256    29.482  22.691    6.791     a4[1]  sum[176]   
9       256  512  256x512    45.140  38.392    6.748  a122[47]  sum[369]   
10      512  512  512x512    46.483  38.378    8.105   a62[15]  sum[356]   

    Delay_MAX  IC_MAX  CELL_MAX  Delay_MEAN    IC_MEAN  CELL_MEAN  
0